In [2]:
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns


# Models
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42


In [3]:
import polars as pl
print(pl.__version__)


1.34.0


Due to the big size of the dataset we use scan_parquet instead of read_parquet so we don't load all the data instantly

In [4]:
df = pl.scan_parquet("processed_github_features.parquet")

In [ ]:
type(df)        
# df.shape     
# df.head()      
df.schema


Split train and test datasets

In [4]:
# add row index
df_idx = df.with_row_index("idx")

# 80% train, 20% test
train = df_idx.filter(pl.col("idx") % 5 != 0).drop("idx")
test  = df_idx.filter(pl.col("idx") % 5 == 0).drop("idx")

train.head



<bound method LazyFrame.head of <LazyFrame at 0x21DEA66E650>>

In [5]:
# total number of rows
n = df.select(pl.len()).collect().item()

# cutoff index
cutoff = int(0.8 * n)

# add row index
df_idx = df.with_row_index("idx")

# split
train = df_idx.filter(pl.col("idx") < cutoff).drop("idx")
test  = df_idx.filter(pl.col("idx") >= cutoff).drop("idx")


In polars the data is retrieved form the LazyFrame only when .collect() is called

In [6]:
train_df = train.collect()

In [7]:
test_df  = test.collect()

In [ ]:
# DROP_COLS = ["repo_name", "day"]
# target = "total_stars_scaled"

# X_train = train_df.drop(DROP_COLS + [target])
# y_train = train_df.get_column(target)

# X_test  = test_df.drop(DROP_COLS + [target])
# y_test  = test_df.get_column(target)


Adjusting the dataset row numbers because of long wait time for training models

In [8]:
train_small = (
    train
    .with_row_index("_idx")
    .with_columns(pl.col("_idx").hash().alias("_h"))
    .sort("_h")
    .head(200_000)
    .drop(["_idx", "_h"])
    .collect()
)




In [9]:
test_small = (
    test
    .with_row_index("_idx")
    .with_columns(pl.col("_idx").hash().alias("_h"))
    .sort("_h")
    .head(50_000)
    .drop(["_idx", "_h"])
    .collect()
)



In [10]:
DROP_COLS = ["repo_name", "day"]
target = "total_stars_scaled"

X_train = train_small.drop(DROP_COLS + [target])
y_train = train_small.get_column(target)

X_test  = test_small.drop(DROP_COLS + [target])
y_test  = test_small.get_column(target)

Model definition

In [11]:

lin_reg_model = LinearRegression()
ridge_reg_model = Ridge(alpha=1.0)
rand_f_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=RANDOM_STATE
    )
grad_boost_model =  GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    )

xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

In [12]:
lin_reg_model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [13]:
ridge_reg_model.fit(X_train, y_train)

,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [14]:
rand_f_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [15]:
grad_boost_model.fit(X_train, y_train)


,loss,'squared_error'
,learning_rate,0.05
,n_estimators,200
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [16]:
xgb_model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [17]:
y_pred_lin   = lin_reg_model.predict(X_test)

print(y_pred_lin)

[0.222939   0.2118807  0.2180856  ... 0.21119666 0.2157273  0.21113776]


In [18]:
y_pred_ridge = ridge_reg_model.predict(X_test)

print(y_pred_ridge)

[0.44549501 0.2200089  0.38747441 ... 0.02075987 0.37882188 0.01876355]


In [19]:
y_pred_rf    = rand_f_model.predict(X_test)
print(y_pred_rf)

[0.45026477 0.21356387 0.39138305 ... 0.0617338  0.37964728 0.        ]


In [20]:
y_pred_gb    = grad_boost_model.predict(X_test)
print(y_pred_gb)

[4.49992638e-01 2.13561257e-01 3.91829178e-01 ... 6.17482514e-02
 3.78881071e-01 2.61966909e-05]


In [21]:

y_pred_xgb = xgb_model.predict(X_test)
print(y_pred_xgb)

[ 4.5064828e-01  2.1356335e-01  3.9069337e-01 ...  6.1788965e-02
  3.8010457e-01 -1.7593242e-05]


In [22]:
def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

In [23]:
results_lin = evaluate(y_test, y_pred_lin),
results_ridge = evaluate(y_test, y_pred_ridge),
results_rfr = evaluate(y_test, y_pred_rf),
results_grad_boost =  evaluate(y_test, y_pred_gb),
results_xgb_boost =  evaluate(y_test, y_pred_xgb),


In [24]:
print(results_lin)

({'MAE': 0.16265296936035156, 'RMSE': np.float64(0.19608605738289298), 'R2': 0.1704883575439453},)


In [25]:
print(results_ridge)

({'MAE': 0.008592968361515496, 'RMSE': np.float64(0.01375894436751239), 'R2': 0.9959158725456494},)


In [26]:
print(results_rfr)

({'MAE': 1.3501943704575514e-05, 'RMSE': np.float64(0.00014548439632369373), 'R2': 0.9999995433726817},)


In [27]:
print(results_grad_boost)

({'MAE': 0.0002893302913195222, 'RMSE': np.float64(0.0006244415108738627), 'R2': 0.9999915877418962},)


In [28]:
print(results_xgb_boost)

({'MAE': 0.00038181335548870265, 'RMSE': np.float64(0.001901789192433176), 'R2': 0.999921977519989},)


The results look too good